# 지시를 따르게 만들기 (SFT)

사전학습된 모델은 **다음 단어를 이어 쓸 줄만** 압니다. "요약해줘"라고 하면
요약하는 게 아니라 그 문장 뒤에 그럴듯한 말을 잇습니다.

**지시를 따르게** 만드는 것이 Supervised Fine-Tuning입니다.
방법은 단순합니다 — **"이렇게 물으면 이렇게 답한다"**는 예시를 잔뜩 보여줍니다.

## 무엇을 하게 되나

1. **어제 만든 데이터**를 불러옵니다 (2일차 Amazon 실습 산출물)
2. 데이터를 **대화 형식**으로 바꿉니다
3. **LoRA**로 일부만 학습시킵니다
4. 학습한 모델에 실제로 물어봅니다

## 어제와 오늘이 이어집니다

2일차에 상품 설명에서 요약을 뽑아 `instruction`/`output` 쌍을 만들었습니다.
그게 오늘의 학습 데이터입니다. **남의 데이터가 아니라 직접 만든 것**으로 학습합니다.

그리고 어제 프롬프트만으로 점수를 얼마나 올렸는지 기억해 두세요.
오늘은 같은 일을 **학습으로** 합니다. 마지막에 그 둘을 비교하게 됩니다.

## 환경 세팅

In [ ]:
%pip install -q -U transformers datasets trl peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# (셀 2 에서 한 번에 설치하므로 삭제)

## 1. 데이터 — 어제 만든 것을 불러온다

두 파일을 합쳐 읽습니다.

| 파일 | 무엇 |
|---|---|
| `assets/amazon_ko_sft.jsonl.gz` | 강사가 미리 만든 것 (상품 100개, 약 800건) |
| `data/amazon_ko_sft.mine.jsonl` | **여러분이** 2일차에 만든 것 (상품 10개) |

두 파일에는 **같은 상품이 들어 있습니다** — 배포본이 상품 100개, 여러분 것이 그중 앞 10개.
그래서 완전히 같은 쌍은 하나만 남깁니다.

다만 실제로 걸리는 것은 거의 없습니다. **생성이 비결정적**이라 같은 상품이라도
문장이 조금씩 다르게 나오기 때문입니다. 그건 오히려 도움이 됩니다 —
같은 입력에 대한 여러 표현을 보는 셈이니까요.

파일이 없으면 공개 데이터셋(KULLM)으로 대체하되 **크게 알립니다.**
폴백인 줄 모르고 "내 데이터로 학습했다" 고 오해하면 안 되니까요.

> 경로를 계산하는 코드가 붙어 있는 이유가 있습니다. Jupyter는 노트북이 있는 폴더를
> 작업 디렉터리로 잡습니다. 2일차 노트북이 쓴 파일을 3일차 노트북이 읽으려면
> **양쪽이 같은 곳을 봐야** 합니다.


In [ ]:
import os
from pathlib import Path

from datasets import load_dataset


def _repo_root():
    """cwd 에서 위로 올라가며 repo 루트를 찾는다. setup_vessl.sh 를 표지로 쓴다."""
    p = Path.cwd().resolve()
    return next((c for c in [p, *p.parents] if (c / "setup_vessl.sh").exists()), None)


_root = _repo_root()
DATA_DIR = Path(os.environ["HPC_DATA"]) if "HPC_DATA" in os.environ else (
    _root / "data" if _root else None)
ASSETS_DIR = (_root / "assets") if _root else None

# 어제 직접 만든 것 + 강사가 미리 만들어 둔 것
paths = [p for p in [
    ASSETS_DIR / "amazon_ko_sft.jsonl.gz" if ASSETS_DIR else None,
    DATA_DIR / "amazon_ko_sft.mine.jsonl" if DATA_DIR else None,
] if p is not None and p.exists()]

if paths:
    raw_datasets = load_dataset("json", data_files=[str(p) for p in paths])

    # 어제 직접 만든 10건은 강사 사전생성본 안에 이미 들어 있다.
    # 그대로 합치면 앞쪽 상품이 두 번 학습된다. 같은 (instruction, output) 은 하나만 남긴다.
    from datasets import Dataset, DatasetDict

    _rows, _seen = [], set()
    for r in raw_datasets["train"]:
        key = (r["instruction"], r["output"])
        if key not in _seen:
            _seen.add(key)
            _rows.append({"instruction": r["instruction"], "output": r["output"]})
    _dup = len(raw_datasets["train"]) - len(_rows)
    raw_datasets = DatasetDict({"train": Dataset.from_list(_rows)})
    if _dup:
        print(f"  겹치는 {_dup}건은 하나로 합쳤습니다")
    print(f"내가 만든 데이터로 학습합니다 — {len(raw_datasets['train']):,}건")
    for p in paths:
        print(f"  · {p.name}")
else:
    # ★ 조용히 넘어가면 폴백인 줄 모른다. 크게 알린다.
    print("=" * 60)
    print("★ 생성 데이터가 없어 남의 데이터(kullm-v2)로 대체합니다.")
    print("  2일차 HPC_Amazon요약실습 을 먼저 돌리면")
    print("  직접 만든 데이터로 학습할 수 있습니다.")
    print("=" * 60)
    raw_datasets = load_dataset("nlpai-lab/kullm-v2")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.75k [00:00<?, ?B/s]

(…)-00000-of-00001-21df739eb88d711e.parquet:   0%|          | 0.00/12.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21155 [00:00<?, ? examples/s]

In [ ]:
from datasets import DatasetDict

# 데이터가 얼마나 있을지 모르므로 건수로 자르지 않고 비율로 나눈다.
# (고정 인덱스로 잘랐다가 평가셋이 0건이 된 적이 있다)
n_all = len(raw_datasets["train"])
n_train = min(1000, int(n_all * 0.9))
n_test = min(50, n_all - n_train)

indices = range(0, n_train)
test_indices = range(n_train, n_train + n_test)
print(f"전체 {n_all:,}건 → 학습 {n_train:,} / 평가 {n_test}")

dataset_dict = {"train": raw_datasets["train"].select(indices),
                "test": raw_datasets["train"].select(test_indices)}

raw_datasets = DatasetDict(dataset_dict)
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output', 'url'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['instruction', 'output', 'url'],
        num_rows: 50
    })
})

## 2. 대화 형식으로 바꾸기

`instruction` / `output` 쌍을 **주고받는 대화**로 바꿉니다.

```
{"instruction": "요약해줘...", "output": "..."}
        ↓
[{"role": "user",      "content": "요약해줘..."},
 {"role": "assistant", "content": "..."}]
```

왜 이렇게 하느냐면, 실제로 쓸 때도 이 형태로 물어보기 때문입니다.
**학습할 때와 쓸 때의 형식이 같아야** 모델이 헷갈리지 않습니다.

이 원칙은 앞으로 계속 나옵니다. 형식이 어긋나면 학습은 정상적으로 끝나는데
막상 써보면 이상한 답이 나옵니다. **조용히 실패하는** 종류입니다.

In [ ]:
def build_messages(example):
    # KULLM 은 Alpaca 계열이라 instruction 외에 input 컬럼이 있다.
    # input 이 있는 행을 합쳐주지 않으면 "다음 문장을 요약하세요" 처럼
    # 지시만 남고 정작 대상이 빠진 학습 예시가 된다.
    user = example["instruction"]
    if example.get("input"):
        user = f'{user}\n\n{example["input"]}'

    messages = [
        {"role": "user", "content": user},
        {"role": "assistant", "content": example["output"]}
    ]

    return dict(messages=messages)

raw_datasets = raw_datasets.map(build_messages)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [ ]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output', 'url', 'messages'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['instruction', 'output', 'url', 'messages'],
        num_rows: 50
    })
})

In [ ]:
messages = raw_datasets["train"][0]["messages"]
for message in messages:
  role = message["role"]
  content = message["content"]
  print('{0:20}:  {1}'.format(role, content))

user                :  양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?
assistant           :  양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. 

식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.

 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? 

고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.


In [ ]:
example = raw_datasets["train"][0]
print(example.keys())

dict_keys(['instruction', 'output', 'url', 'messages'])


### 형식이 제대로 만들어졌는지 봅니다

`role`과 `content`가 짝을 이루는지, `user` 다음에 `assistant`가 오는지 확인합니다.

여기가 어긋나면 뒤에서 전부 어긋납니다. 그런데 **에러가 나지 않아서** 학습이
끝날 때까지 모릅니다. 그래서 눈으로 한 번 봅니다.

## 3. 챗 템플릿 — Base 모델에는 대화 형식이 없다

모델 이름 끝의 **`-Base`**를 보세요. 사전학습만 하고 대화 학습은 안 한 모델입니다.
그래서 `<|user|>` 같은 **대화 표시를 모릅니다.**

우리가 직접 정해서 넣어줍니다. 그것이 `DEFAULT_CHAT_TEMPLATE`입니다.
Jinja 문법이라 한 줄로 붙어 있어 읽기 어렵지만, 하는 일은 이겁니다.

```
[{"role":"user","content":"안녕"},
 {"role":"assistant","content":"반가워"}]
        ↓  템플릿을 거치면
<|user|>
안녕<|endoftext|>
<|assistant|>
반가워<|endoftext|>
```

**역할을 표시하는 특수 문자열로 감싸는 것**이 전부입니다.
모델은 이 표시를 보고 "여기서부터 내가 답할 차례"를 배웁니다.

### 나머지 두 줄

| 코드 | 왜 |
|---|---|
| `pad_token_id = eos_token_id` | Base 모델엔 패딩 토큰이 없다. 문장 끝 토큰을 대신 쓰는 관례 |
| `model_max_length > 100_000`이면 2048 | Qwen3의 기본 컨텍스트가 지나치게 길다. 실습용으로 줄인다 |

> **`-Instruct` 모델은 이미 템플릿을 갖고 있습니다.** 그때는 이 셀이 필요 없습니다.
> 오히려 덮어쓰면 모델이 학습한 것과 어긋나 성능이 떨어집니다.


In [ ]:
from transformers import AutoTokenizer

model_id = "Qwen/Qwen3-0.6B-Base"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# set pad_token_id equal to the eos_token_id if not set
if tokenizer.pad_token_id is None:
  tokenizer.pad_token_id = tokenizer.eos_token_id

# Set reasonable default for models without max length
if tokenizer.model_max_length > 100_000:
  tokenizer.model_max_length = 2048

# Set chat template
DEFAULT_CHAT_TEMPLATE = "{% for message in messages %}\n{% if message['role'] == 'user' %}\n{{ '<|user|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'system' %}\n{{ '<|system|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'assistant' %}\n{{ '<|assistant|>\n'  + message['content'] + eos_token }}\n{% endif %}\n{% if loop.last and add_generation_prompt %}\n{{ '<|assistant|>' }}\n{% endif %}\n{% endfor %}"
tokenizer.chat_template = DEFAULT_CHAT_TEMPLATE

## 챗 템플릿 적용과 길이 확인

위에서 정한 형식을 실제로 씌웁니다. `messages`가 한 덩어리 문자열(`text`)이 됩니다.

`tokenize=False`인 것에 주의하세요. 여기서는 **문자열까지만** 만듭니다.
토큰으로 바꾸는 것은 학습할 때 TRL이 알아서 합니다.

빈 `system` 메시지를 앞에 끼워 넣는 것도 보입니다. 템플릿이 세 역할을 다 기대하는데
데이터에는 `system`이 없어서, **형식을 맞추려고** 넣습니다.

그리고 **토큰 길이를 잽니다.** 아래 셀의 주석을 꼭 읽어보세요 —
이 확인을 건너뛰면 어떻게 조용히 실패하는지 적혀 있습니다.

In [ ]:
import re
import random
from multiprocessing import cpu_count

def apply_chat_template(example, tokenizer):
    messages = example["messages"]
    # We add an empty system message if there is none
    if messages[0]["role"] != "system":
        messages.insert(0, {"role": "system", "content": ""})
    example["text"] = tokenizer.apply_chat_template(messages, tokenize=False)

    return example

column_names = list(raw_datasets["train"].features)
raw_datasets = raw_datasets.map(apply_chat_template,
                                num_proc=cpu_count(),
                                fn_kwargs={"tokenizer": tokenizer},
                                remove_columns=column_names,
                                desc="Applying chat template",)

# create the splits
train_dataset = raw_datasets["train"]
eval_dataset = raw_datasets["test"]

# ── 학습 전에 토큰 길이를 본다 ──────────────────────────────────
# ★ max_length 에 걸리면 뒤가 잘린다. 그런데 잘리는 쪽은 **정답**이다 —
#   instruction 이 앞에 오기 때문이다. 그러면 loss 는 낮게 나오면서
#   아무것도 안 배우는 상태가 되고, 에러도 나지 않는다.
#   조용히 실패하는 종류라 반드시 확인하고 넘어간다.
_lens = sorted(len(tokenizer.encode(t)) for t in train_dataset["text"])
_limit = tokenizer.model_max_length
_over = sum(1 for n in _lens if n > _limit)

print()
print(f"토큰 길이 — 중앙값 {_lens[len(_lens)//2]:,} · "
      f"상위10% {_lens[int(len(_lens)*0.9)]:,} · 최대 {_lens[-1]:,}")
print(f"상한 {_limit:,} 초과: {_over}건 ({_over/len(_lens):.1%})")
if _over:
    print("  ★ 초과분은 뒤가 잘립니다. instruction 이 앞이므로 정답이 날아갑니다.")
    print("    Amazon 노트북의 MAX_SRC 를 줄이거나 model_max_length 를 늘리세요.")
else:
    print("  → 잘리는 예시 없음")

for index in random.sample(range(len(raw_datasets["train"])), 3):
  print(f"Sample {index} of the processed training set:\n\n{raw_datasets['train'][index]['text']}")

Applying chat template (num_proc=8):   0%|          | 0/1000 [00:00<?, ? examples/s]

Applying chat template (num_proc=8):   0%|          | 0/50 [00:00<?, ? examples/s]

Sample 6 of the processed training set:

<|system|>
<|endoftext|>
<|user|>
야구 글러브의 종류와 포지션별 특징은 무엇인가요?<|endoftext|>
<|assistant|>
보통 4칸으로 그물처럼 되어있는 것이 내야글러브이며, 1루를 제외한 내야 전 포지션에서 사용 가능합니다. 그물처럼 촘촘히 되어있는 것이 외야글러브이며, 우익수, 중견수, 좌익수에서 사용 가능합니다. 투수 글러브는 그립이 보이지 않도록 막혀 있습니다. 포수와 1루수는 글러브가 아닌 미트를 사용합니다.<|endoftext|>

Sample 141 of the processed training set:

<|system|>
<|endoftext|>
<|user|>
전자레인지로 생쌀 밥을 할 수 있는 방법이 있을까요?<|endoftext|>
<|assistant|>
전자레인지로 생쌀 밥을 할 수 있는 방법이 있습니다. 불린 쌀과 물, 그리고 머그컵과 접시를 사용하여 7분 내로 간편하게 밥을 할 수 있습니다. 머그컵에 불린 쌀과 물을 섞어 넣고, 접시를 얹은 후 전자렌지에서 3분간 요리합니다. 이후 1분 정도 쉬고 다시 3분간 요리합니다. 이 때 조금 뜨거운 물이 넘치는 것을 방지하기 위해서 접시를 들춰줍니다. 불린 쌀은 10분 이상, 적어도 20~30분 정도를 불리는 것이 좋으며, 쌀과 물의 비율은 3:4 정도가 적당합니다. 사용하는 용기의 종류에 따라 물과 쌀의 양, 요리 시간이 달라질 수 있으니 참고해주세요. 프리미엄 밥맛을 원한다면 전용 뚜껑이 있는 용기를 사용하시면 좋습니다.<|endoftext|>

Sample 975 of the processed training set:

<|system|>
<|endoftext|>
<|user|>
사막 사람들은 어떤 일을 하며 살까요? 사하라사막에서는 살아남기 위해 무슨 일을 하나요? 이집트는 전체가 사막인가요?<|endoftext|>
<|assistant|>
사막 지역에도 현재는 사람들이 살

In [ ]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['text'],
        num_rows: 50
    })
})

## 4. LoRA — 전부 학습시키지 않는다

0.6B 모델도 전부 학습시키려면 메모리가 만만치 않습니다. 그리고 대부분의 경우
**그럴 필요도 없습니다.**

**LoRA**는 원래 가중치를 얼려두고, 옆에 **작은 행렬 두 개**를 붙여 그것만 학습합니다.
학습이 끝나면 그 작은 것만 저장하면 됩니다 — 수백 MB가 아니라 수십 MB입니다.

```
원래 가중치 W  (얼림, 학습 안 함)
      +
   A × B      (작다. 이것만 학습)
```

### 값 읽는 법

| 값 | 뜻 |
|---|---|
| `r=64` | 붙이는 행렬의 **크기**. 클수록 표현력이 늘고 무거워진다 |
| `lora_alpha=16` | 그 결과를 얼마나 **세게 반영**할지 |
| `target_modules` | 어디에 붙일지. 여기서는 **어텐션 4곳**(q·k·v·o) |
| `lora_dropout=0.1` | 과적합 억제 |

`print_trainable_parameters()`를 꼭 보세요. **전체의 몇 %만 학습하는지** 나옵니다.
그 숫자가 LoRA를 쓰는 이유를 한눈에 보여줍니다.

> 어텐션에만 붙이고 피드포워드(`gate/up/down_proj`)는 뺐습니다. 관례적인 선택이고,
> 붙이면 표현력은 늘지만 무거워집니다. 정답이 있는 값은 아닙니다.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
)

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=64,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

trainable params: 18,350,080 || all params: 614,400,000 || trainable%: 2.9867


In [ ]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

## 5. 학습 설정

1일차 분류 실습과 값이 꽤 다릅니다. 이유가 있습니다.

| | 분류 (1일차) | SFT |
|---|---|---|
| 학습률 | `2e-5` | **`5e-4`** — 25배 |
| 배치 | 16 | 2 × 누적 8 = **유효 16** |

**학습률이 큰 것은 LoRA 때문입니다.** 원래 가중치는 얼려 뒀고 새로 붙인 작은 행렬만
바닥부터 배우므로, 조심스럽게 갈 이유가 없습니다.

**배치를 쪼개는 것은 메모리 때문입니다.** 한 번에 16개를 올리면 GPU 메모리가 넘칩니다.
2개씩 8번 계산해서 **기울기를 모았다가 한 번에 갱신**합니다. 결과는 배치 16과 같고
메모리만 1/8입니다. `gradient_accumulation_steps`가 그것입니다.

`gradient_checkpointing`도 같은 목적입니다 — 중간 계산 결과를 저장하지 않고
필요할 때 다시 계산합니다. 메모리를 아끼고 시간을 씁니다.

> `use_reentrant: False`는 PEFT와 gradient checkpointing을 함께 쓸 때 필요한
> 관용구입니다. 빼면 경고가 나거나 기울기가 흐르지 않습니다.

In [ ]:
from trl import SFTTrainer, SFTConfig

output_dir = 'data/sft_model'

# based on config
training_args = SFTConfig(
    fp16=True,
    do_eval=True,
    eval_strategy="epoch",
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=5.0e-04,
    log_level="info",
    logging_steps=5,
    logging_strategy="steps",
    lr_scheduler_type="cosine",
    max_steps=-1,
    num_train_epochs=3,
    output_dir=output_dir,
    per_device_eval_batch_size=1,
    per_device_train_batch_size=2,
    save_strategy="epoch",
    save_total_limit=1,
    seed=42,
    dataset_text_field="text",
    packing=False,
    max_length=tokenizer.model_max_length,
    report_to="tensorboard"
)

trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
    )

PyTorch: setting up devices
loading file vocab.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-Base/snapshots/11214f7f3465775dcce23c3752ecea5a42ee0ddc/vocab.json
loading file merges.txt from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-Base/snapshots/11214f7f3465775dcce23c3752ecea5a42ee0ddc/merges.txt
loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-Base/snapshots/11214f7f3465775dcce23c3752ecea5a42ee0ddc/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-Base/snapshots/11214f7f3465775dcce23c3752ecea5a42ee0ddc/tokenizer_config.json
loading file chat_template.jinja from cache at None
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Using auto half precision back

## 학습

돌려놓고 기다립니다. 몇 분 걸립니다.

`loss`가 내려가는지 보세요. 내려가지 않으면 학습률이나 데이터 형식을 의심합니다.
**너무 빨리 0에 가까워지는 것도 좋은 신호가 아닙니다** — 데이터가 너무 적거나
같은 것을 반복해서 외우는 중일 수 있습니다.

In [ ]:
train_result = trainer.train()

The following columns in the training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text. If text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 1,000
  Num Epochs = 3
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 8
  Total optimization steps = 186
  Number of trainable parameters = 18,350,080
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,1.968100,2.052623
2,1.860900,2.052208


The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text. If text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 50
  Batch size = 1
Saving model checkpoint to data/test_model/checkpoint-63
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-Base/snapshots/11214f7f3465775dcce23c3752ecea5a42ee0ddc/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "max_position_embeddings": 32768,
  "max_window_layers": 28,
  "model_type": "qwen3",
  "num_attention_heads": 16,
  "num_hidden_layers": 28,
  "num_k

In [ ]:
trainer.model.save_pretrained(output_dir)
trainer.processing_class.save_pretrained(output_dir)

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-Base/snapshots/11214f7f3465775dcce23c3752ecea5a42ee0ddc/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "max_position_embeddings": 32768,
  "max_window_layers": 28,
  "model_type": "qwen3",
  "num_attention_heads": 16,
  "num_hidden_layers": 28,
  "num_key_value_heads": 8,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 1000000,
  "sliding_window": null,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.51.3",
  "use_cache": true,
  "use_sliding_window": false,
  "vocab_size": 151936
}

Trainer.tokenizer is now deprecated. You should use Trainer.processing_c

('data/test_model/tokenizer_config.json',
 'data/test_model/special_tokens_map.json',
 'data/test_model/vocab.json',
 'data/test_model/merges.txt',
 'data/test_model/added_tokens.json',
 'data/test_model/tokenizer.json')

## 써보기

학습한 어댑터를 다시 불러와 실제로 물어봅니다.

**챗 템플릿을 다시 넣는 것**에 주의하세요. 템플릿은 우리가 코드로 지정한 것이라
모델 파일에 자동으로 따라가지 않습니다. 불러온 뒤 다시 지정해야
**학습할 때와 같은 형식**으로 물어볼 수 있습니다.

앞에서 말한 "학습할 때와 쓸 때의 형식이 같아야 한다"가 여기서 지켜집니다.

질문을 바꿔 넣어보세요. 학습 데이터가 **상품 요약**이었으니 그쪽은 잘하고,
전혀 다른 질문에는 어색할 겁니다. 그것이 파인튜닝의 성질입니다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

output_dir = 'data/sft_model'
tokenizer = AutoTokenizer.from_pretrained(output_dir)
model = AutoModelForCausalLM.from_pretrained(output_dir, device_map="auto")

loading file vocab.json
loading file merges.txt
loading file tokenizer.json
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading file chat_template.jinja
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-Base/snapshots/11214f7f3465775dcce23c3752ecea5a42ee0ddc/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "max_position_embeddings": 32768,
  "max_window_layers": 28,
  "model_type": "qwen3",
  "num_attention_heads": 16,
  "num_hidden_layers": 28,
  "num_key_value_heads": 8,
  "rms_norm_eps"

In [ ]:
tokenizer.chat_template = DEFAULT_CHAT_TEMPLATE

In [ ]:
import torch

messages = [
    {"role": "system", "content": ""},
    {"role": "user", "content": "HPC라는게 무엇인가요?"},
]

# transformers 5 에서 apply_chat_template 은 BatchEncoding 을 돌려준다.
# 예전처럼 generate(input_ids=...) 로 넘기면 AttributeError 가 난다.
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
    return_tensors="pt", return_dict=True,
).to("cuda")

outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.9,
        top_p=0.95
)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


<|system|>

<|user|>
HPC라는게 무엇인가요?
<|assistant|>
HPC(High Performance Computing)는 지속적인 업그레이드를 통해 유연하고 대량의 처리량을 가진 포괄적인 인프라와 인력으로 구성된 인프라입니다. 이 인프라는 컴퓨터 시스템, 클라우드, 센서, 시스템 등 다양한 분야에서 사용됩니다. 대표적인 인프라는 수명주기 10년 이상의 노드를 가진 하드웨어와 인력과 콜렉트됩니다. 이 인프라에서, 여러 계층에서 복잡한 처리 및 데이터 처리를 수행하는 것이 가능한 상태입니다. HPC의 장점 중 하나는 데이터 처리 성능의 빠른 증가입니다. 대개적으로 수명주기 5년 이상의 인프라를 구성하고 있습니다.


## 마무리

지시를 따르는 모델을 만들었습니다.

- 사전학습 모델은 **이어 쓸 줄만** 압니다. 지시를 따르게 하려면 예시를 보여줘야 합니다
- **Base 모델에는 대화 형식이 없습니다.** 우리가 정해서 넣어줍니다
- **학습할 때와 쓸 때의 형식이 같아야** 합니다. 어긋나면 조용히 실패합니다
- **LoRA**로 전체의 일부만 학습해도 충분합니다
- 학습 전에 **토큰 길이를 확인**합니다. 잘리면 정답이 날아갑니다

### 여기서 끝이 아닙니다

SFT는 **"이렇게 답해라"**를 가르칩니다. 그런데 정답이 하나로 정해지지 않는
작업에서는 이것만으로 부족합니다. 요약은 여러 가지가 다 맞을 수 있으니까요.

그럴 때 쓰는 것이 **선호 학습**입니다 — "이 답이 저 답보다 낫다"를 가르칩니다.
다음 실습(DPO)에서 **방금 학습한 이 모델을 이어받아** 그것을 해봅니다.